# DecompDiff - Colab sweep 5 (the third stream)

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> **GPU**.

The stream the code calls "residual" has always carried **raw x_t** - nothing is subtracted,
so it is an identity path, and the trend and seasonal streams duplicate part of what it already
supplies. `real_residual=True` feeds it `x_t - trend - season` instead, so the three streams
partition the signal exactly (`trend + season + remainder == x_t`).

Why this might matter on fMRI: the remainder holds **12.1%** of the window energy at L=24, against
0.1-1.1% on the trend-dominated datasets, and it is mid- to high-frequency content - the band the
generated samples are measurably short of. Giving it a dedicated stream targets that deficit
directly.

The flag adds **no parameters** (262,322 either way on fMRI) and leaves the `state_dict`
unchanged, so any difference is attributable to the change itself rather than to capacity.

One consequence worth knowing: with the flag **off**, the stream tags `--R` and `---` are the
same model (both feed `res_pe(res_input_proj(x_t))` into the fusion stack, bit for bit). With the
flag **on** they finally differ - `--R` is remainder-only, `---` is still raw x_t.

| Cell | What it does |
|------|--------------|
| 1 | Check GPU |
| 2 | Clone the repo |
| 3 | Set paths |
| 4-5 | wandb install + login |
| 6 | Imports |
| 7 | Loss, LR schedule, metrics |
| 8 | `run_experiment` |
| 9 | Shared settings |
| 10+ | Experiments: fMRI third stream, stock control, --R stream ablation |

## 1. Setup

In [ ]:
# -- Check GPU --------------------------------------------------------------
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"Memory  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# -- Clone repo from GitHub (always fresh) ---------------------------------
# Needs the auxiliary-head version of DecompDiff/models/decompDiff.py, so make
# sure that change is pushed before running this.
import os
REPO_URL = "https://github.com/Ardameliksah/DecompDiff.git"
REPO_DIR = "/content/DecompDiff-colab"
!rm -rf {REPO_DIR}
!git clone --depth 1 {REPO_URL} {REPO_DIR}
!ls {REPO_DIR}/libs/MyCode

In [ ]:
import os, sys
from pathlib import Path
assert Path(REPO_DIR).exists(), f"Folder not found: {REPO_DIR}"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())
print("has DecompDiff:", Path("DecompDiff").exists(), "| has libs/MyCode:", Path("libs/MyCode").exists())

In [ ]:
# -- Install missing packages ----------------------------------------------
!pip install -q wandb

In [ ]:
# -- Weights & Biases login -------------------------------------------------
import os, wandb
try:
    from google.colab import userdata
    key = userdata.get("WANDB_API_KEY")
    if key:
        os.environ["WANDB_API_KEY"] = key
except Exception:
    pass
if os.environ.get("WANDB_API_KEY"):
    wandb.login(key=os.environ["WANDB_API_KEY"])
else:
    wandb.login()
print("wandb:", wandb.__version__)

## 2. Imports & experiment function

In [ ]:
# -- Imports ----------------------------------------------------------------
import math, os, random, sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
import wandb

REPO = Path(REPO_DIR)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "libs"))

from DecompDiff.models.decompDiff_res import DecompDiffRes
from DecompDiff.models.diffusion  import GaussianDiffusion
from DecompDiff.config.stocks_config import Config
from DecompDiff.data.datasets import make_loaders
from MyCode.eval_metrics import (discriminative_score, predictive_score,
                                 vds_score, fdds_score, correlational_score)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("REPO:", REPO, "| device:", DEVICE)

In [ ]:
def compute_loss(model, diffusion, x_0, loss_type="mse", prediction_type="x0",
                 use_loss_weight=True, lam_trend=0.0, lam_season=0.0):
    """Diffusion loss, optionally plus per-stream component supervision.

    lam_trend == lam_season == 0  ->  identical to sweep 3, and the auxiliary
    heads are never even called. Otherwise each stream's head is scored against
    its own component of the CLEAN sample x_0, taken from the same
    SeriesDecomposition the model runs internally.

    Returns (loss, parts) where parts holds the raw auxiliary terms for logging.
    """
    B = x_0.shape[0]
    t = torch.randint(0, diffusion.num_timesteps, (B,), device=x_0.device)
    x_t, noise = diffusion.q_sample(x_0, t)

    want_aux = (lam_trend > 0.0 or lam_season > 0.0) and getattr(model, "use_aux_heads", False)
    if want_aux:
        model_out, aux = model(x_t, t, return_aux=True)
    else:
        model_out, aux = model(x_t, t), {}

    target = x_0 if prediction_type == "x0" else noise
    loss_fn = F.l1_loss if loss_type == "l1" else F.mse_loss
    loss = loss_fn(model_out, target, reduction="none")

    parts = {}
    if want_aux:
        with torch.no_grad():
            trend_gt, season_gt = model.decomp(x_0)      # ground-truth components
        if lam_trend > 0.0 and "trend" in aux:
            l_tr = loss_fn(aux["trend"], trend_gt, reduction="none")
            loss = loss + lam_trend * l_tr
            parts["aux_trend"] = float(l_tr.mean().item())
        if lam_season > 0.0 and "season" in aux:
            l_se = loss_fn(aux["season"], season_gt, reduction="none")
            loss = loss + lam_season * l_se
            parts["aux_season"] = float(l_se.mean().item())

    # the per-timestep weight is applied LAST, after the auxiliary terms
    loss = loss.mean(dim=[1, 2])
    if use_loss_weight:
        loss = loss * diffusion.loss_weight[t]
    return loss.mean(), parts


# LR schedule modes (same as sweep 3)
#   "cosine"  -> one warmup+cosine decay over the whole run (default)
#   "restart" -> that cycle restarting every lr_epochs epochs
#   "flat"    -> constant LR after warmup
LR_MODES = ("cosine", "restart", "flat")


def build_lr_scheduler(optimizer, steps_per_epoch, num_epochs, lr_epochs=None,
                       warmup_steps=100, base_lr=1e-4, eta_min=1e-6):
    cycle_epochs = lr_epochs or num_epochs
    cycle_steps  = max(1, steps_per_epoch * cycle_epochs)
    warm         = max(0, min(warmup_steps, cycle_steps - 1))
    floor        = eta_min / base_lr

    def lr_lambda(step):
        s = step % cycle_steps
        if s < warm:
            return 1e-3 + (1.0 - 1e-3) * (s / max(1, warm))
        prog = (s - warm) / max(1, cycle_steps - warm)
        return floor + (1.0 - floor) * 0.5 * (1.0 + math.cos(math.pi * prog))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# Which evaluations run at each eval point
ALL_METRICS = ("disc", "pred", "vds", "fdds", "corr")
METRICS     = ("disc", "pred")


def compute_inline_metrics(model, diffusion, real_loader, device, num_steps=50,
                           prediction_type="x0", metrics=None, n_iterations=3,
                           disc_iterations=2000, pred_iterations=5000):
    """Sample from the model and compute ONLY the metrics named in `metrics`."""
    sel = tuple(metrics) if metrics is not None else tuple(METRICS)
    bad = [m for m in sel if m not in ALL_METRICS]
    if bad:
        raise ValueError(f"unknown metric(s) {bad}; valid options: {ALL_METRICS}")
    if not sel:
        return None

    model.eval()
    batches = [b.cpu().numpy() for b in real_loader]
    real_CL = np.concatenate(batches, axis=0)
    N = real_CL.shape[0]
    chunk, chunks = 256, []
    with torch.no_grad():
        for start in range(0, N, chunk):
            bs = min(chunk, N - start)
            chunks.append(
                model.sample(diffusion, batch_size=bs, num_steps=num_steps, eta=0.0,
                             prediction_type=prediction_type).cpu()
            )
    fake_CL = torch.cat(chunks, dim=0).numpy()
    real_m = ((real_CL.transpose(0, 2, 1) + 1.0) * 0.5).astype(np.float32)
    fake_m = ((fake_CL.transpose(0, 2, 1) + 1.0) * 0.5).astype(np.float32)

    out = {}
    try:
        if "disc" in sel:
            print("  computing discriminative score...")
            ds, accs = [], []
            for i in range(n_iterations):
                d, a = discriminative_score(real_m, fake_m,
                                            iterations=disc_iterations, device=device)
                ds.append(d); accs.append(a)
                print(f"    run {i+1}/{n_iterations}: disc={d:.4f} test_acc={a:.4f}")
            out["disc_score"]     = float(np.mean(ds))
            out["disc_score_std"] = float(np.std(ds))
            out["test_acc"]       = float(np.mean(accs))
        if "pred" in sel:
            print("  computing predictive score...")
            ps = []
            for i in range(n_iterations):
                p = predictive_score(real_m, fake_m,
                                     iterations=pred_iterations, device=device)
                ps.append(p)
                print(f"    run {i+1}/{n_iterations}: MAE={p:.4f}")
            out["pred_mae"]     = float(np.mean(ps))
            out["pred_mae_std"] = float(np.std(ps))
        if "vds" in sel:
            out["vds"] = vds_score(real_m, fake_m)
        if "fdds" in sel:
            out["fdds"] = fdds_score(real_m, fake_m)
        if "corr" in sel:
            out["correlational_score"] = correlational_score(real_m, fake_m)
    except Exception as e:
        print(f"  [metrics] failed: {e}"); model.train(); return None

    model.train()
    return out

In [ ]:
def run_experiment(dataset, window_length, num_epochs, num_layers=1,
                   num_fusion_layers=1, hidden_dim=64, batch_size=None,
                   use_trend=True, use_season=True, use_residual=True,
                   real_residual=False,
                   lam_trend=0.0, lam_season=0.0,
                   prediction_type="x0", use_loss_weight=True, sample_steps=50,
                   eval_every=500, metrics=None, lr_epochs=None,
                   learning_rate=None, lr_min=None, lr_mode=None,
                   seed=None, run_name=None):
    """Winning objective (MSE - predict x0 - loss weight ON) on any dataset.

    `lam_trend` / `lam_season` weight the auxiliary component losses. Both 0
    (the default) builds the model WITHOUT the heads, so the run is identical
    to sweep 3 in parameters, state_dict and behaviour.

    `real_residual` switches the third stream from raw x_t (an identity path,
    the original behaviour) to the true remainder x_t - trend - season, so the
    three streams partition the signal instead of the third duplicating it.
    It adds no parameters, so checkpoints stay interchangeable.

    `prediction_type` ("x0" or "eps"), `use_loss_weight` and `sample_steps`
    expose the training objective and the DDIM step count. The defaults
    ("x0", True, 50) are exactly the sweep-3 configuration.
    """
    LOSS   = "mse"
    PRED   = prediction_type
    WEIGHT = use_loss_weight
    if PRED not in ("x0", "eps"):
        raise ValueError(f'prediction_type must be "x0" or "eps", got {PRED!r}')
    sel_metrics   = tuple(metrics) if metrics is not None else tuple(METRICS)
    use_aux_heads = (lam_trend > 0.0) or (lam_season > 0.0)

    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    cfg = Config()
    cfg.model.sequence_length = window_length
    cfg.training.num_epochs   = num_epochs
    cfg.data.dataset          = dataset
    cfg.model.num_layers        = num_layers
    cfg.model.num_fusion_layers = num_fusion_layers
    cfg.model.hidden_dim        = hidden_dim
    cfg.model.use_trend         = use_trend
    cfg.model.use_season        = use_season
    cfg.model.use_residual      = use_residual
    cfg.training.loss_type       = LOSS
    cfg.training.prediction_type = PRED
    cfg.training.use_loss_weight = WEIGHT
    if batch_size is not None:
        cfg.training.batch_size = batch_size
    if learning_rate is not None:
        cfg.training.learning_rate = learning_rate
    bs      = cfg.training.batch_size
    base_lr = cfg.training.learning_rate

    if lr_mode is None:
        lr_mode = "restart" if lr_epochs else "cosine"
    if lr_mode not in LR_MODES:
        raise ValueError(f"lr_mode must be one of {LR_MODES}, got {lr_mode!r}")
    if lr_mode == "restart" and not lr_epochs:
        raise ValueError('lr_mode="restart" needs lr_epochs=<cycle length in epochs>')
    if lr_mode != "restart" and lr_epochs:
        raise ValueError(f'lr_epochs only applies to lr_mode="restart" (got {lr_mode!r})')
    if lr_mode == "flat" and lr_min is not None:
        raise ValueError('lr_mode="flat" holds the LR at learning_rate; drop lr_min')
    floor_lr     = base_lr if lr_mode == "flat" else (
        lr_min if lr_min is not None else base_lr / 100.0)
    cycle_epochs = lr_epochs if lr_mode == "restart" else num_epochs
    n_cycles     = math.ceil(num_epochs / cycle_epochs)

    stag = ("T" if use_trend else "-") + ("S" if use_season else "-") + ("R" if use_residual else "-")

    train_loader, _, ds = make_loaders(
        dataset, batch_size=bs, window=window_length,
        train_ratio=cfg.data.train_split, neg_one_to_one=cfg.data.neg_one_to_one,
        per_window=cfg.data.per_window_norm, num_workers=cfg.data.num_workers,
        pin_memory=False, data_root=cfg.data.data_root,
        sine_num=cfg.data.sine_num, sine_dim=cfg.data.sine_dim, seed=cfg.data.sine_seed,
    )
    cfg.model.input_channels = ds.num_features

    aux_tag = f"-aux{lam_trend:g}_{lam_season:g}" if use_aux_heads else ""
    lr_tag  = ((f"-lrE{cycle_epochs}" if lr_mode == "restart" else "")
               + ("-lrflat" if lr_mode == "flat" else "")
               + (f"-lr{base_lr:.0e}" if learning_rate is not None else "")
               + (f"-min{floor_lr:.0e}" if lr_min is not None else "")
               + (f"-s{seed}" if seed is not None else ""))
    step_tag = (f"-st{sample_steps}" if sample_steps != 50 else "") + ("-rres" if real_residual else "")
    run_name = run_name or (
        f"decompdiff-{dataset}-L{window_length}-H{hidden_dim}"
        f"-NL{num_layers}-NF{num_fusion_layers}-{LOSS}-{PRED}-w{int(WEIGHT)}-bs{bs}-{stag}"
        f"-E{num_epochs}{aux_tag}{lr_tag}{step_tag}"
    )

    wandb.init(
        project="decompdiff-sweep5", group="third-stream", name=run_name,
        tags=[dataset, f"bs{bs}", f"NL{num_layers}", f"NF{num_fusion_layers}",
              "loss-mse", f"pred-{PRED}", f"w{int(WEIGHT)}", f"streams-{stag}",
              "aux-on" if use_aux_heads else "aux-off"],
        config={**cfg.model.__dict__, **cfg.diffusion.__dict__, **cfg.training.__dict__,
                "dataset": dataset, "device": DEVICE, "eval_every": eval_every,
                "eval_metrics": list(sel_metrics),
                "lr_mode": lr_mode, "lr_epochs": cycle_epochs,
                "lr_cycles": n_cycles, "lr_min": floor_lr, "seed": seed,
                "lam_trend": lam_trend, "lam_season": lam_season,
                "use_aux_heads": use_aux_heads, "sample_steps": sample_steps,
                "real_residual": real_residual},
    )

    model = DecompDiffRes(
        input_channels=cfg.model.input_channels, sequence_length=window_length,
        hidden_dim=hidden_dim, num_heads=cfg.model.num_heads, num_layers=num_layers,
        num_fusion_layers=num_fusion_layers, mlp_ratio=cfg.model.mlp_ratio,
        dropout=cfg.model.dropout, freq_dim=cfg.model.freq_dim,
        use_trend=use_trend, use_season=use_season, use_residual=use_residual,
        real_residual=real_residual, use_aux_heads=use_aux_heads,
    ).to(DEVICE)
    diffusion = GaussianDiffusion(
        num_timesteps=cfg.diffusion.num_timesteps, beta_start=cfg.diffusion.beta_start,
        beta_end=cfg.diffusion.beta_end, noise_schedule=cfg.diffusion.noise_schedule,
        device=DEVICE,
    ).to(DEVICE)

    counts = model.get_parameter_count()
    print(f"[{run_name}] batches={len(train_loader)} channels={cfg.model.input_channels} "
          f"batch_size={bs} metrics={sel_metrics}")
    print(f"[{run_name}] lr [{lr_mode}]: {base_lr:.1e} -> {floor_lr:.1e}, "
          f"{cycle_epochs}-epoch cycle x {n_cycles}")
    third_stream = "x_t - trend - season" if real_residual else "raw x_t"
    print(f"[{run_name}] third stream: {third_stream}")
    print(f"[{run_name}] objective: {LOSS} / predict {PRED} / loss_weight {WEIGHT} "
          f"| sampling {sample_steps} DDIM steps")
    print(f"[{run_name}] aux_heads={use_aux_heads} lam_trend={lam_trend} lam_season={lam_season}")
    print(f"[{run_name}] model_dim={model.model_dim} params={counts['total']:,}")
    wandb.config.update({"total_params": counts["total"], "model_dim": model.model_dim},
                        allow_val_change=True)

    optimizer = AdamW(model.parameters(), lr=base_lr,
                      weight_decay=cfg.training.weight_decay, betas=(0.9, 0.999))
    scheduler = build_lr_scheduler(optimizer, len(train_loader), num_epochs,
                                   lr_epochs=lr_epochs,
                                   warmup_steps=cfg.training.warmup_steps,
                                   base_lr=base_lr, eta_min=floor_lr)

    ckpt_dir = REPO / f"DecompDiff/output/checkpoints/{run_name}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_every = max(100, num_epochs // 5)

    for epoch in range(num_epochs):
        model.train(); epoch_loss = 0.0; aux_sums = {}
        for batch in train_loader:
            x_0 = batch.to(DEVICE)
            optimizer.zero_grad()
            loss, parts = compute_loss(model, diffusion, x_0, LOSS,
                                       prediction_type=PRED, use_loss_weight=WEIGHT,
                                       lam_trend=lam_trend, lam_season=lam_season)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.training.gradient_clip_val)
            optimizer.step(); scheduler.step(); epoch_loss += loss.item()
            for k, v in parts.items():
                aux_sums[k] = aux_sums.get(k, 0.0) + v

        nb = len(train_loader)
        train_loss = epoch_loss / nb
        current_lr = optimizer.param_groups[0]["lr"]
        log = {"train_loss": train_loss, "lr": current_lr, "epoch": epoch + 1,
               "lr_cycle": epoch // cycle_epochs + 1}
        for k, v in aux_sums.items():
            log[k] = v / nb

        if (epoch + 1) % save_every == 0 or (epoch + 1) == num_epochs:
            torch.save({"epoch": epoch + 1, "train_loss": train_loss,
                        "model_state_dict": model.state_dict(),
                        "config": {"model": cfg.model.__dict__, "window_length": window_length,
                                   "dataset": dataset, "prediction_type": PRED,
                                   "use_loss_weight": WEIGHT, "loss_type": LOSS,
                                   "lam_trend": lam_trend, "lam_season": lam_season,
                                   "use_aux_heads": use_aux_heads}},
                       ckpt_dir / f"checkpoint_ep{epoch+1}.pt")

        if (epoch + 1) % eval_every == 0:
            print(f"  [epoch {epoch+1}] metrics {sel_metrics}...")
            eval_out = compute_inline_metrics(model, diffusion, train_loader, DEVICE,
                                              num_steps=sample_steps, prediction_type=PRED,
                                              metrics=sel_metrics)
            if eval_out is not None:
                log.update(eval_out)
                print("  " + "  ".join(f"{k}={v:.4f}" for k, v in eval_out.items()))
        wandb.log(log)
        aux_str = ("  " + "  ".join(f"{k}={v/nb:.5f}" for k, v in aux_sums.items())) if aux_sums else ""
        print(f"[{run_name}] ep {epoch+1:4d}/{num_epochs} train={train_loss:.5f} lr={current_lr:.2e}{aux_str}")

    print(f"[{run_name}] done. ckpt -> {ckpt_dir}")
    wandb.finish()
    return model, diffusion

In [ ]:
# -- Shared settings --------------------------------------------------------
EPOCHS     = 2000
EVAL_EVERY = 500
WINDOW     = 24

# METRICS (set in the loss cell above) decides which evaluations run.
# Default ("disc", "pred"). Pass metrics=ALL_METRICS on a cell for the full set.
print(f"epochs={EPOCHS} eval_every={EVAL_EVERY} window={WINDOW} metrics={METRICS}")

## 3. The third stream: identity path vs true remainder

Five runs, all window 24 / 2000 epochs / 200 DDIM steps. Each has its control either already run
in sweep 3-4 or paired here. Run names carry `-rres`, and every run prints
`third stream: x_t - trend - season` or `raw x_t` so you can confirm at a glance.

| # | Dataset | Config | Pairs with |
|---|---|---|---|
| 1 | fmri | eps + w1, H100, NL1/NF1, **remainder** | the same config in sweep 4 |
| 2 | fmri | x0 + w1, H64, NL1/NF1, **remainder** | sweep 4 section 3 cell A |
| 3 | fmri | eps + w1, H100, NL2/NF2, **remainder** | the NL2/NF2 run |
| 4-5 | stock | x0 + w1, H64, identity vs **remainder** | each other |

Cell 2 is the most informative: it tests the change under the **original** objective, so an
improvement there is not tangled up with the eps switch.

In [ ]:
# -- fmri | eps + w1 | H100 | NL1/NF1 | TRUE REMAINDER | window 24 | 2000 ep | 200 steps --
# A/B against the same config with real_residual=False (your current best run).
run_experiment(dataset="fmri", window_length=24, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=100,
               prediction_type="eps", use_loss_weight=True, sample_steps=200,
               real_residual=True,
               eval_every=EVAL_EVERY)

import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# -- fmri | x0 + w1 | H64 | NL1/NF1 | TRUE REMAINDER | window 24 | 2000 ep | 200 steps --
# Pairs with section 3 cell A, so it tests the change under the ORIGINAL objective too.
run_experiment(dataset="fmri", window_length=24, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=64,
               prediction_type="x0", use_loss_weight=True, sample_steps=200,
               real_residual=True,
               eval_every=EVAL_EVERY)

import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# -- fmri | eps + w1 | H100 | NL2/NF2 | TRUE REMAINDER | window 24 | 2000 ep | 200 steps --
run_experiment(dataset="fmri", window_length=24, num_epochs=EPOCHS,
               num_layers=2, num_fusion_layers=2, hidden_dim=100,
               prediction_type="eps", use_loss_weight=True, sample_steps=200,
               real_residual=True,
               eval_every=EVAL_EVERY)

import gc; gc.collect(); torch.cuda.empty_cache()

### stock control - does the change hurt trend-dominated data?

On stock the remainder is only 0.2% of the signal energy, so the true remainder is nearly empty
and the third stream gives up the full-signal copy it used to carry. Run both cells: if the
treatment is clearly worse here, the flag is fMRI-specific and should be reported that way.

In [ ]:
# -- stock | x0 + w1 | H64 | NL1/NF1 | identity path (CONTROL) | window 24 | 2000 ep | 200 steps --
run_experiment(dataset="stock", window_length=24, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=64,
               prediction_type="x0", use_loss_weight=True, sample_steps=200,
               eval_every=EVAL_EVERY)

import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# -- stock | x0 + w1 | H64 | NL1/NF1 | TRUE REMAINDER | window 24 | 2000 ep | 200 steps --
run_experiment(dataset="stock", window_length=24, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=64,
               prediction_type="x0", use_loss_weight=True, sample_steps=200,
               real_residual=True,
               eval_every=EVAL_EVERY)

import gc; gc.collect(); torch.cuda.empty_cache()

## 4. Stream ablation under the true remainder

With `real_residual=False` the tags `--R` and `---` are **the same model**: both feed
`res_pe(res_input_proj(x_t))` into the fusion stack, bit for bit, because nothing is ever
subtracted. Any difference between those two rows in the old ablation was seed noise.

With `real_residual=True` they separate:

| Tag | flag off | flag on |
|---|---|---|
| `--R` | raw x_t only | **x_t - trend - season only** |
| `---` | raw x_t only | raw x_t only (the fallback is unchanged) |

So `--R` with the flag on finally tests what its name always claimed: can the model work from
the remainder alone? The decomposition is computed even though the trend and season streams are
off, and the remainder goes through the same projection + positional encoding into fusion.

Both cells use the current best fMRI objective so they are comparable to section 3 cell 1, which
is the `TSR` version of the same configuration.

In [ ]:
# -- fmri | --R with TRUE REMAINDER: remainder is the ONLY stream | eps + w1 | H100 | 2000 ep --
# trend and season streams are off, but the decomposition still runs; the third stream
# carries x_t - trend - season through proj + PE into the fusion stack.
run_experiment(dataset="fmri", window_length=24, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=100,
               prediction_type="eps", use_loss_weight=True, sample_steps=200,
               use_trend=False, use_season=False, use_residual=True,
               real_residual=True,
               eval_every=EVAL_EVERY)

import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# -- fmri | --R with the identity path (CONTROL, equals ---) | eps + w1 | H100 | 2000 ep --
# Same stream configuration, flag off: the third stream carries raw x_t.
# This is the control for the cell above, and is also exactly what --- computes.
run_experiment(dataset="fmri", window_length=24, num_epochs=EPOCHS,
               num_layers=1, num_fusion_layers=1, hidden_dim=100,
               prediction_type="eps", use_loss_weight=True, sample_steps=200,
               use_trend=False, use_season=False, use_residual=True,
               eval_every=EVAL_EVERY)

import gc; gc.collect(); torch.cuda.empty_cache()

## 5. (optional) Back up checkpoints to Drive

In [ ]:
# Colab runtimes are ephemeral - checkpoints under /content are lost on disconnect.
from google.colab import drive; drive.mount("/content/drive")
import shutil
from pathlib import Path
dst = "/content/drive/MyDrive/DecompDiff_output_sweep5"
shutil.copytree(str(Path(REPO_DIR) / "DecompDiff" / "output"), dst, dirs_exist_ok=True)
print("copied ->", dst)